# 01 -- Build the modelling base table

**What this notebook does (plain English):** This turns the assembled data into
the table every model will use. For each loan it records three things that drive
all later numbers:

- **Default** -- did the loan go badly wrong? (180+ days late, or it ended in a
  loss event such as a foreclosure sale.)
- **EAD (Exposure at Default)** -- how much money was still owed when it defaulted.
- **LGD (Loss Given Default)** -- of that exposure, how much was *actually lost*
  after the property was sold and costs/recoveries settled. This is computed from
  Freddie Mac's **real loss fields**, which is the centrepiece of the project.

**Headline result:** average loss-given-default is far worse in the downturn
(~55-58% in 2007/2008) than in the calm year (~25% in 2015).

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the cached loan-level table from notebook 00.
import numpy as np
import pandas as pd
from src import definitions as d
from src.output import save_csv
df = pd.read_parquet('data/processed/loan_level.parquet')
print(df.shape)

(850000, 57)


In [3]:
# Realised loss and LGD are only defined for defaulted loans that DISPOSED
# (reached a final sale). For everyone else LGD is left blank, on purpose.
df['realised_loss'] = np.where(df['disposed'], d.realised_loss(df), np.nan)
lgd_raw = df['realised_loss'] / df['ead'].replace(0, np.nan)
df['lgd'] = np.where(df['disposed'], d.winsorise_lgd(lgd_raw), np.nan)

In [4]:
# Sanity check: reconcile our computed loss against Freddie Mac's own
# actual_loss_calculation field (their number is stored as a negative loss).
rec = df[df['disposed']].dropna(subset=['actual_loss_calculation'])
rec = rec[rec['actual_loss_calculation'] != 0]
corr = np.corrcoef(-rec['actual_loss_calculation'], d.realised_loss(rec))[0, 1]
print(f'loss reconciliation correlation vs dataset field: {corr:.3f}')

loss reconciliation correlation vs dataset field: 0.991


In [5]:
# Add simple risk bands we will reuse in the EDA and models.
df['credit_score_band'] = pd.cut(df['credit_score'], [0, 620, 660, 700, 740, 780, 851],
                                 right=False, labels=['<620', '620-659', '660-699', '700-739', '740-779', '780+'])
df['ltv_band'] = pd.cut(df['original_ltv'], [0, 60, 70, 80, 90, 200],
                        right=False, labels=['<60', '60-69', '70-79', '80-89', '90+'])

### The workout period (input to the discounting step)

When a loan defaults the money is **not** lost all at once -- the lender works
through foreclosure and sale over many months, and a dollar recovered years later
is worth less than a dollar today. The **workout period** is how long that takes:
from the **default month** to the **disposition month**.

- The **default month** (`default_period`) already comes straight from the data --
  the first month the loan was 180+ days late or hit a loss event.
- The **disposition month** (`disposition_period`) uses the **exact zero-balance
  effective date** where Freddie Mac records it, and falls back automatically to
  the **last servicing month** (`last_period`) when that date is missing.

`months_to_resolution` is that gap in whole months, and it is only meaningful for
**disposed defaults** (NaN everywhere else). Together with `original_interest_rate`
it is one of the two inputs the economic-loss discounting in notebook 01 /
`definitions.economic_loss()` will use (see LGD alignment task P1-1).

In [6]:
# Disposition month: exact zero-balance date if loaded, else last servicing month.
if 'disposition_period' in df.columns:
    df['disposition_period'] = df['disposition_period'].fillna(df['last_period'])
else:
    df['disposition_period'] = df['last_period']

# Workout length in months, only meaningful for disposed defaults.
df['months_to_resolution'] = np.where(
    df['disposed'],
    d.months_between(df['default_period'], df['disposition_period']),
    np.nan,
)

### Nominal vs *economic* loss -- discounting (P1-1)

The framework's very first definition of LGD (CRE36.76 / APS 113 Att D LGD para 1)
is **economic loss**, which *must* include "material discount effects". The `lgd`
column above is **nominal** -- it adds up the dollars lost without caring *when*
they were lost. But a mortgage workout takes months or years (`months_to_resolution`),
and a dollar recovered years after default is worth less than a dollar today.

So we add a second, framework-aligned figure, `lgd_econ`:

- We treat the net recovery (sale proceeds + insurance + other recoveries, **minus**
  foreclosure costs) as a single cash flow arriving `months_to_resolution` months
  after default, and **discount it back** to the default date.
- The discount rate is the **facility's own contractual rate** -- the first choice
  in **APG 113 para 122 / Table 8** -- i.e. `original_interest_rate` converted to a
  monthly rate `r_m = (rate/100)/12`.

Discounting shrinks the present value of the recovery, so **economic loss is always
>= nominal loss**, and the gap is widest for the longest workouts. The original
nominal `lgd` (and its ~0.99 reconciliation) is kept untouched.

In [7]:
# Economic (discounted) loss + LGD -- the framework's actual LGD definition.
# Kept SEPARATE from nominal `lgd`; reduces to it when the workout is instant.
df['economic_loss'] = np.where(df['disposed'], d.economic_loss(df), np.nan)
lgd_econ_raw = df['economic_loss'] / df['ead'].replace(0, np.nan)
df['lgd_econ'] = np.where(df['disposed'], d.winsorise_lgd(lgd_econ_raw), np.nan)
print('avg nominal LGD : {:.4f}'.format(df.loc[df['disposed'], 'lgd'].mean()))
print('avg economic LGD: {:.4f}  (>= nominal, as discounting requires)'.format(
    df.loc[df['disposed'], 'lgd_econ'].mean()))

avg nominal LGD : 0.5281
avg economic LGD: 0.5802  (>= nominal, as discounting requires)


### IFRS 9 view vs **APRA regulatory-capital view** of LGD (P1-2, P1-3)

The numbers so far answer *"what was actually lost economically?"* -- the **IFRS 9 /
accounting** question, where every real recovery (including mortgage insurance) counts.
APRA's **capital** rules deliberately answer a more conservative question and we keep
them in a **separate** column, `lgd_apra`, never overwriting the IFRS 9 number:

- **No LMI credit (APS 113 Att B para 23).** You may **not** use lender's-mortgage-
  insurance recoveries inside a retail-mortgage LGD. We rebuild the loss with
  `include_mi=False`, then apply the permitted **20% LGD reduction** on the high-LVR
  (LVR > 80) loans that actually carry LMI -- the relief the rule grants in place of
  the recovery.
- **LGD floor (APS 113 Att B paras 19-24, Tables 6-7).** A regulatory minimum LGD is
  then applied -- **20%** for retail residential mortgages where own-LGD estimates are
  not approved (stated as our assumption).

Removing the MI recovery *raises* the loss, so for MI-covered high-LVR loans
`lgd_apra >= lgd`. The floor is a backstop on top. This column is the APRA-view
overlay only; the IFRS 9 `lgd` / `lgd_econ` are left exactly as computed.

In [8]:
# APRA regulatory-capital view: MI excluded, 20% high-LVR+LMI reduction, 20% floor.
loss_no_mi = np.where(df['disposed'], d.economic_loss(df, include_mi=False), np.nan)
lgd_no_mi = d.winsorise_lgd(loss_no_mi / df['ead'].replace(0, np.nan))
mi_present = pd.to_numeric(df['mi_pct'], errors='coerce').fillna(0) > 0
high_lvr = pd.to_numeric(df['original_ltv'], errors='coerce') > 80
# 20% LGD reduction where LVR>80 and LMI is in place (APS 113 Att B para 23).
lgd_apra = np.where(mi_present & high_lvr, lgd_no_mi * (1 - 0.20), lgd_no_mi)
lgd_apra = d.apply_lgd_floor(lgd_apra, floor=0.20)  # APS 113 retail mortgage floor
df['lgd_apra'] = np.where(df['disposed'], lgd_apra, np.nan)
mi_hi = df['disposed'] & mi_present & high_lvr
print('MI-covered high-LVR disposed defaults:', int(mi_hi.sum()))
print('  avg IFRS 9 lgd : {:.4f}'.format(df.loc[mi_hi, 'lgd'].mean()))
print('  avg APRA lgd   : {:.4f}  (>= IFRS 9: MI recovery removed)'.format(
    df.loc[mi_hi, 'lgd_apra'].mean()))

MI-covered high-LVR disposed defaults: 4076
  avg IFRS 9 lgd : 0.4604
  avg APRA lgd   : 0.5562  (>= IFRS 9: MI recovery removed)


In [9]:
# Keep one clean analysis row per loan and cache it for later notebooks.
base_cols = [
    'loan_sequence_number', 'vintage_year', 'credit_score', 'original_ltv',
    'original_cltv', 'original_dti', 'original_interest_rate', 'original_loan_term',
    'original_upb', 'loan_purpose', 'occupancy_status', 'channel', 'number_of_borrowers',
    'mi_pct', 'credit_score_band', 'ltv_band', 'ever_default', 'default_within_12m',
    'default_within_12m_90dpd', 'disposed', 'max_delinq_status',
    'ead', 'realised_loss', 'lgd',
    'default_period', 'disposition_period', 'months_to_resolution',
    'economic_loss', 'lgd_econ', 'lgd_apra',
]
base = df[base_cols].copy()
base.to_parquet('data/processed/analysis_base.parquet')
print('analysis base:', base.shape)

analysis base: (850000, 30)


### One-year PD target -- `default_within_12m` (PD-1)

The headline `ever_default` flag asks *"did this loan ever go bad in the history we
observed?"* That is the right lens for lifetime loss and for the LGD population, but it
is **not** how a PD is defined for capital. The framework's foundational definition
(CRE36.63 / APS 113 Att D PD para 2) is a **long-run average of one-year default
rates** -- so the PD target must be measured over a **fixed 12-month window** for every
loan, regardless of how long we happened to watch it.

`default_within_12m` is exactly that: a default (180+ DPD or a credit-event) occurring
within the **first 12 months** of the loan's life (from `loan_age`). Because older books
are observed for far longer than recent ones, their *observed-to-date* rates are not
comparable; the fixed one-year window puts all 17 vintages on the **same footing**.

`ever_default` and `disposed` are kept **unchanged** -- the PD notebooks switch to the
one-year flag, while the lifetime-EL view and the LGD work keep using `ever_default`.

In [10]:
# Sanity-check the one-year PD target: it must be a SUBSET of ever_default, and
# the 12-month default rate is now measured over the same window for all vintages.
assert (df['default_within_12m'] & ~df['ever_default']).sum() == 0, '12m default must imply ever_default'
pd_target = df.groupby('vintage_year').agg(
    loans=('loan_sequence_number', 'size'),
    ever_default_rate=('ever_default', 'mean'),
    one_year_default_rate=('default_within_12m', 'mean'),
).reset_index().round(4)
print(pd_target.to_string(index=False))
print('one-year default is a strict subset of ever-default:',
      bool((df['default_within_12m'] <= df['ever_default']).all()))

 vintage_year  loans  ever_default_rate  one_year_default_rate
         2006  50000             0.1205                 0.0086
         2007  50000             0.1374                 0.0126
         2008  50000             0.0735                 0.0094
         2009  50000             0.0227                 0.0018
         2010  50000             0.0219                 0.0016
         2011  50000             0.0204                 0.0019
         2012  50000             0.0223                 0.0021
         2013  50000             0.0235                 0.0016
         2014  50000             0.0238                 0.0016
         2015  50000             0.0242                 0.0014
         2016  50000             0.0273                 0.0010
         2017  50000             0.0361                 0.0023
         2018  50000             0.0343                 0.0013
         2019  50000             0.0335                 0.0079
         2020  50000             0.0119                

In [11]:
# Sanity-check the new workout-length field on disposed defaults only.
wr = df.loc[df['disposed'], 'months_to_resolution']
print('disposed defaults with a usable months_to_resolution:', int(wr.notna().sum()))
print('min / median / max months: {:.0f} / {:.0f} / {:.0f}'.format(
    wr.min(), wr.median(), wr.max()))
print('share resolved in 0 months:', round(float((wr == 0).mean()), 4))
# Confirm the field is blank for everyone who did NOT dispose-as-default.
print('non-disposed loans with a non-NaN value (should be 0):',
      int(df.loc[~df['disposed'], 'months_to_resolution'].notna().sum()))

disposed defaults with a usable months_to_resolution: 13466
min / median / max months: 0 / 14 / 183
share resolved in 0 months: 0.0545
non-disposed loans with a non-NaN value (should be 0): 0


In [12]:
# Results table: default rate and average LGD by vintage (downturn vs calm),
# now showing nominal vs economic (discounted) and the APRA-view LGD side by side.
tbl = df.groupby('vintage_year').agg(
    loans=('loan_sequence_number', 'size'),
    default_rate=('ever_default', 'mean'),
    disposed_defaults=('disposed', 'sum'),
    avg_lgd=('lgd', 'mean'),
    avg_lgd_econ=('lgd_econ', 'mean'),
    avg_lgd_apra=('lgd_apra', 'mean'),
    median_lgd=('lgd', 'median'),
    avg_ead=('ead', 'mean'),
).reset_index().round(4)
save_csv(tbl, 'outputs/tables/01_default_lgd_by_vintage.csv')
tbl

,vintage_year,loans,default_rate,disposed_defaults,avg_lgd,avg_lgd_econ,avg_lgd_apra,median_lgd,avg_ead
0,2006,50000,0.1205,4066,0.5818,0.6356,0.6479,0.5740,179409.7700
1,2007,50000,0.1374,4479,0.5783,0.6323,0.6438,0.5631,186177.0150
2,2008,50000,0.0735,2134,0.5441,0.5986,0.6134,0.5131,198213.9893
3,2009,50000,0.0227,543,0.4189,0.4702,0.4978,0.3591,180958.0616
4,2010,50000,0.0219,496,0.3975,0.4394,0.4728,0.3604,178322.7674
5,2011,50000,0.0204,379,0.3854,0.4280,0.4643,0.3134,168727.3786
6,2012,50000,0.0223,379,0.3385,0.3731,0.4194,0.2942,171948.0959
7,2013,50000,0.0235,305,0.3690,0.4015,0.4463,0.3045,164819.9005
8,2014,50000,0.0238,202,0.3564,0.4015,0.4483,0.2282,170015.0329
9,2015,50000,0.0242,136,0.2464,0.3033,0.3635,0.1678,197949.9096


**Reading the table:** both the chance of default *and* the severity of
loss when it happens are much worse in the crisis vintages -- the two effects
compound, which is exactly why a downturn hurts a mortgage book so much.

Across the new LGD columns: **`avg_lgd_econ` >= `avg_lgd`** in every vintage
(discounting the recovery raises the loss), and **`avg_lgd_apra`** sits higher
again because it strips out mortgage-insurance recoveries and imposes the 20%
regulatory floor. The three columns are deliberately kept separate: nominal IFRS 9,
economic IFRS 9, and the conservative APRA capital view.